# Tool Use with Claude (Cho Claude dùng Công cụ)

**Khóa: Claude with the Anthropic API (Anthropic Skilljar)**

### Vấn đề
Bản thân Claude **không thể**:
- Tính toán chính xác các phép tính lớn 🧮
- Biết thông tin thời gian thực (thời tiết, tỉ giá...) 🌤️
- Tra cứu dữ liệu riêng của bạn (database, API nội bộ) 🗄️
- Thực hiện hành động (gửi email, đặt lịch...) 📧

### Giải pháp: Tool Use (Function Calling)
Bạn **mô tả các công cụ** cho Claude. Khi cần, Claude sẽ **yêu cầu gọi công cụ** với tham số phù hợp. **Code của bạn** thực thi công cụ và trả kết quả về → Claude dùng kết quả để trả lời.

> 🔑 Quan trọng: **Claude KHÔNG tự chạy công cụ.** Nó chỉ *yêu cầu*; bạn là người thực thi và đưa kết quả lại.


In [ ]:
!pip install anthropic

In [ ]:
import anthropic, json

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

## 1. Luồng hoạt động của Tool Use (5 bước)

```
1. Bạn gửi request KÈM danh sách tools
2. Claude trả về stop_reason="tool_use" + khối tool_use (tên + tham số)
3. Code của bạn THỰC THI công cụ
4. Bạn gửi kết quả lại dưới dạng tool_result
5. Claude dùng kết quả -> trả lời cuối cùng (stop_reason="end_turn")
```

Bước 2→4 có thể lặp lại nhiều lần (Claude gọi nhiều công cụ) → gọi là **vòng lặp agentic**.


## 2. Định nghĩa một công cụ (Tool Definition)

Mỗi công cụ gồm 3 phần:
- **`name`**: tên công cụ
- **`description`**: mô tả RÕ RÀNG công cụ làm gì & khi nào dùng (Claude dựa vào đây để quyết định gọi)
- **`input_schema`**: JSON Schema mô tả các tham số đầu vào

Ta định nghĩa 2 công cụ: tính toán và tra cứu thời tiết (giả lập).


In [ ]:
tools = [
    {
        "name": "calculator",
        "description": "Thực hiện một phép tính số học. Dùng khi cần tính toán chính xác.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Biểu thức toán học hợp lệ trong Python, ví dụ '23 * 17 + 5'",
                }
            },
            "required": ["expression"],
        },
    },
    {
        "name": "get_weather",
        "description": "Lấy thông tin thời tiết hiện tại của một thành phố.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Tên thành phố, ví dụ 'Hà Nội'"}
            },
            "required": ["city"],
        },
    },
]

## 3. Viết hàm thực thi công cụ (phía bạn)

Đây là code THỰC SỰ chạy khi Claude yêu cầu. Trong thực tế, `get_weather` sẽ gọi API thật; ở đây ta giả lập bằng dữ liệu cứng.


In [ ]:
def calculator(expression):
    # CẢNH BÁO: eval chỉ dùng cho demo. Thực tế cần kiểm tra/sandbox kỹ.
    return str(eval(expression))

def get_weather(city):
    fake_data = {
        "Hà Nội": "28°C, nắng nhẹ",
        "Đà Nẵng": "31°C, có mây",
        "TP.HCM": "33°C, oi bức",
    }
    return fake_data.get(city, "Không có dữ liệu cho thành phố này.")

# Bộ điều phối: tên công cụ -> hàm tương ứng
def run_tool(name, tool_input):
    if name == "calculator":
        return calculator(tool_input["expression"])
    elif name == "get_weather":
        return get_weather(tool_input["city"])
    return "Công cụ không tồn tại."

## 4. Xem Claude YÊU CẦU gọi công cụ (1 lượt thủ công)

Gửi câu hỏi cần tính toán. Quan sát `stop_reason` và khối `tool_use`.


In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=[{"role": "user", "content": "23 nhân 17 cộng 100 bằng bao nhiêu?"}],
)

print("stop_reason:", response.stop_reason)   # -> 'tool_use'
print()
for block in response.content:
    if block.type == "text":
        print("[TEXT]:", block.text)
    elif block.type == "tool_use":
        print("[TOOL_USE]")
        print("  id   :", block.id)
        print("  name :", block.name)
        print("  input:", block.input)

## 5. Thực thi & trả kết quả về (tool_result)

Quy tắc bắt buộc:
1. Thêm **toàn bộ** `response.content` vào lịch sử (vai trò `assistant`).
2. Gửi 1 message `user` chứa khối **`tool_result`** — phải có **`tool_use_id`** khớp với `id` ở khối tool_use.


In [ ]:
messages = [
    {"role": "user", "content": "23 nhân 17 cộng 100 bằng bao nhiêu?"},
    {"role": "assistant", "content": response.content},   # giữ nguyên khối tool_use
]

# Thực thi từng tool_use và gom kết quả
tool_results = []
for block in response.content:
    if block.type == "tool_use":
        result = run_tool(block.name, block.input)
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,     # phải khớp id
            "content": result,
        })

messages.append({"role": "user", "content": tool_results})

# Gọi lại Claude với kết quả -> nó trả lời cuối cùng
final = client.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)
print("stop_reason:", final.stop_reason)      # -> 'end_turn'
print(final.content[0].text)

## 6. Vòng lặp Agentic hoàn chỉnh (tự động hóa)

Trong thực tế Claude có thể gọi NHIỀU công cụ liên tiếp. Ta gói toàn bộ vào một **vòng lặp**: cứ khi nào `stop_reason == "tool_use"` thì thực thi & gửi kết quả, lặp đến khi `end_turn`.


In [ ]:
def agent_loop(user_message):
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages
        )

        # Nếu Claude trả lời xong -> dừng
        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")

        # Ngược lại: Claude muốn gọi công cụ
        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  🔧 Gọi {block.name}({block.input})")
                result = run_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })
        messages.append({"role": "user", "content": tool_results})

In [ ]:
# Thử câu hỏi cần NHIỀU công cụ
print("Câu hỏi: Thời tiết Hà Nội thế nào, và 15% của 240 là bao nhiêu?\n")
answer = agent_loop("Thời tiết Hà Nội thế nào, và 15% của 240 là bao nhiêu?")
print("\n💬 Trả lời:", answer)

## 7. Xử lý lỗi trong công cụ (is_error)

Nếu công cụ chạy lỗi, trả `tool_result` kèm `"is_error": True` và mô tả lỗi → Claude sẽ tự điều chỉnh (thử lại hoặc hỏi rõ thêm).


In [ ]:
# Ví dụ khối tool_result báo lỗi
error_result = {
    "type": "tool_result",
    "tool_use_id": "toolu_xxx",
    "content": "Lỗi: không tìm thấy thành phố 'XYZ'. Hãy cung cấp tên hợp lệ.",
    "is_error": True,
}
print(json.dumps(error_result, ensure_ascii=False, indent=2))

## 8. (Nâng cao) Tool Runner — SDK tự lo vòng lặp

SDK có sẵn helper **`tool_runner`** (beta) tự động chạy vòng lặp agentic cho bạn — chỉ cần khai báo công cụ bằng decorator `@beta_tool`, không cần viết loop thủ công.


In [ ]:
from anthropic import beta_tool

@beta_tool
def add(a: int, b: int) -> str:
    """Cộng hai số nguyên.

    Args:
        a: số thứ nhất
        b: số thứ hai
    """
    return str(a + b)

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=1024,
    tools=[add],
    messages=[{"role": "user", "content": "Cộng 123 và 877 giúp tôi."}],
)

# Lặp đến khi xong; in message cuối
for message in runner:
    last = message
print(next(b.text for b in last.content if b.type == "text"))

---
## ✅ Tóm tắt

- **Tool Use** mở rộng năng lực của Claude: tính toán, tra cứu, hành động.
- Luồng 5 bước: gửi tools → Claude yêu cầu `tool_use` → bạn thực thi → trả `tool_result` → Claude trả lời.
- **Claude không tự chạy công cụ** — bạn là người thực thi.
- 3 phần của 1 tool: `name`, `description` (càng rõ Claude càng gọi đúng), `input_schema`.
- Quy tắc nối kết quả: giữ nguyên `response.content`, trả `tool_result` với `tool_use_id` khớp.
- **Vòng lặp agentic**: lặp đến khi `stop_reason == "end_turn"`.
- Lỗi → `is_error: True`. Tự động hóa → `tool_runner`.

**Bài tiếp:** thường là kết hợp Tool Use thành **Agent** hoàn chỉnh, hoặc RAG / Retrieval.
